In [118]:
!pip install pypdf langchain-community

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [119]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("2024-wttc-introduction-to-ai.pdf")

pages = loader.load()

print(pages)

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 1\nINTRODUCTION \nTO ARTIFICIAL \nINTELLIGENCE (AI) \nTECHNOLOGY\nGUIDE FOR TRAVEL & TOURISM LEADERS\nJanuary 2024'), Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n2\n< Contents  |\nCONTENTS\nFOREWORD ........................................................................................

In [120]:
for page in pages:
    print(page.page_content)

INTRODUCTION TO AI
World Travel & Tourism Council
< Contents  | 1
INTRODUCTION 
TO ARTIFICIAL 
INTELLIGENCE (AI) 
TECHNOLOGY
GUIDE FOR TRAVEL & TOURISM LEADERS
January 2024
INTRODUCTION TO AI
World Travel & Tourism Council
2
< Contents  |
CONTENTS
FOREWORD ...........................................................................................................3
INTRODUCTION ..................................................................................................4
Brief history of Artificial Intelligence (AI) 4
What is Artificial Intelligence & Why There is Global Interest Now  7
Algorithms : The Brains of AI 8
Data : The Fuel That Drives AI 12
Computing Power : The Machines Behind AI 16
Types of Artificial Intelligence 21
Generative AI 24
Global Digital Divide & AI Skills Gap 31
QUIZ....................................................................................................................... 34
ANNEX : AI TERMINOLOGY..................................................

In [121]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [122]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

In [123]:
chunks = text_splitter.split_documents(pages)

In [124]:
print("Total chunks:", len(chunks))

print("\nFirst chunk:")
print(chunks[0].page_content)

Total chunks: 156

First chunk:
INTRODUCTION TO AI
World Travel & Tourism Council
< Contents  | 1
INTRODUCTION 
TO ARTIFICIAL 
INTELLIGENCE (AI) 
TECHNOLOGY
GUIDE FOR TRAVEL & TOURISM LEADERS
January 2024


In [125]:
from langchain_huggingface import HuggingFaceEmbeddings

In [126]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [127]:
from langchain_community.vectorstores import FAISS

In [128]:
vectorstore = FAISS.from_documents(chunks, embeddings)

In [129]:
print("Total chunks stored:", vectorstore.index.ntotal)

Total chunks stored: 156


In [130]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [131]:
query = "Explain the three types of Artificial Intelligence in detail."

In [132]:
results = retriever.invoke(query)

In [133]:
for doc in results:
    print(doc.page_content)
    print("----------------")

these AI systems including supervised learning, unsupervised learning and reinforcement learning.
The second useful way to group, or classify, an AI system is based on its capability, which means an AI system 
delivers either artificial narrow intelligence (ANI), artificial general intelligence (AGI), or artificial super 
intelligence (ASI).
----------------
is impossible to accurately predict how AI will shape our lives over the next 5 to 50 years, but the following 
section explains why now is a crucial moment for the future of AI and society.
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary 
defines AI as “the capacity of computers, or other machines, to exhibit intelligent behavio

In [134]:
context = "\n".join(
    [doc.page_content for doc in results]
)

print(context)

these AI systems including supervised learning, unsupervised learning and reinforcement learning.
The second useful way to group, or classify, an AI system is based on its capability, which means an AI system 
delivers either artificial narrow intelligence (ANI), artificial general intelligence (AGI), or artificial super 
intelligence (ASI).
is impossible to accurately predict how AI will shape our lives over the next 5 to 50 years, but the following 
section explains why now is a crucial moment for the future of AI and society.
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary 
defines AI as “the capacity of computers, or other machines, to exhibit intelligent behaviour”. This means A

In [135]:
prompt = f"""
Answer the question using only the following context.
Explain in simple English.

Context:
{context}

Question:
{query}

Answer:
"""

In [136]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [137]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [138]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("LLM Answer:", answer)

LLM Answer: Artificial narrow intelligence (ANI), artificial general intelligence (AGI), or artificial super intelligence (ASI).
